# Round 2: Chinese GP Qualifying Analysis

## Imports and Data Loading

In [ ]:
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fastf1 import plotting
from src.plotset import setup_plot
from src.utils import compute_track_dominance_multi, quali_track_evolution
from src.plotset import plot_track_dominance, plot_quali_track_evolution, save_fig

setup_plot()

In [ ]:
fastf1.Cache.enable_cache('./f1_cache')
fastf1.Cache.get_cache_info()

In [ ]:
session = fastf1.get_session(2026, 2, 'Q')
session.load()

## Track Domination

In [ ]:
driver_colors = {
    'ANT': plotting.get_driver_color(identifier='ANT',session=session),
    'HAM': plotting.get_driver_color(identifier='HAM',session=session)
}

In [ ]:
points, segments, colors = compute_track_dominance_multi(session=session, drivers=['ANT','HAM'], circuit_length=5451, window_size=200, colors_map=driver_colors)

In [ ]:
dominance_plot = plot_track_dominance(points, segments, colors, figsize=(12, 8))

In [ ]:
save_fig(fig=dominance_plot, name='track_dominance', loc='Reel25', trs=True)

## Qualifying Laptime Evolution

In [ ]:
df = quali_track_evolution(session=session)

In [ ]:
evo = plot_quali_track_evolution(df=df)

In [ ]:
save_fig(fig=evo, name='track_evolution', loc='Reel25', trs=True)

## Laptime Delta Analysis

In [ ]:
corners = session.get_circuit_info().corners
rotation = session.get_circuit_info().rotation

In [ ]:
ref_lap = session.laps.pick_drivers('ANT').pick_fastest().get_telemetry().copy()
comp_lap = session.laps.pick_drivers('HAM').pick_fastest().get_telemetry().copy()

In [ ]:
mult = ref_lap.Distance.iloc[-1]/comp_lap.Distance.iloc[-1]

ref_dist = ref_lap.Distance.to_numpy()
ref_time = ref_lap.Time.dt.total_seconds().to_numpy()
comp_dist = (comp_lap.Distance * mult).to_numpy()
comp_time = comp_lap.Time.dt.total_seconds().to_numpy()

In [ ]:
comp_time = np.interp(ref_dist, comp_dist, comp_time)

In [ ]:
delta = comp_time - ref_time

In [ ]:
setup_plot(axeslabel=24, figtitle=28, legendfont=22)

ant_color = plotting.get_driver_color(session=session, identifier='ANT')
ham_color = plotting.get_driver_color(session=session, identifier='HAM')

fig, ax = plt.subplots(2,1,figsize=(15,12),height_ratios=[0.7,0.3],sharex=True)
ax[0].plot(ref_lap['Distance'],ref_lap['Speed'],color=ant_color,linewidth=3, label='ANTONELLI')
ax[0].plot(comp_lap['Distance'],comp_lap['Speed'],color=ham_color,linewidth=3, label='HAMILTON')

ax[0].vlines(x=corners['Distance'], ymin=0, ymax=340,
          linestyles='dotted', colors='grey')
# for _, corner in corners.iterrows():
#     txt = f"{corner['Number']}{corner['Letter']}"
#     ax[0].text(corner['Distance'], 340, txt,
#             va='center_baseline', ha='center', size=15, color="#00FF0D")

# ax[0].set_title('Speed Trace', pad=30)
ax[0].set_xlim(ref_dist[0],ref_dist[-1])
ax[0].set_ylim(0, 350)
ax[0].set_ylabel('Speed (km/h)')
ax[0].legend()
ax[0].grid(visible=False)
# ax02 = ax[0].twinx()
# ax02.set_yticks([])
# ax02.set_ylabel('BK',color="#292929",labelpad=30)

ax[1].plot(ref_lap['Distance'],delta,ls='--',lw=3,color='w')
ax[1].axhline(y=0,lw=1,ls='--',color="#FFFFFF")

ax[1].vlines(x=corners['Distance'], ymin=-1.2, ymax=1.2,
          linestyles='dotted', colors='grey')

ax[1].fill_between(x=(ref_dist[0],ref_dist[-1]),y1=0,y2=1.2,color=ant_color,alpha=0.2)
ax[1].fill_between(x=(ref_dist[0],ref_dist[-1]),y1=-1.2,y2=0,color=ham_color,alpha=0.2)

# ax[1].set_title('Lap Time Delta')
ax[1].set_xlim(ref_dist[0],ref_dist[-1])
ax[1].set_ylim(-1.2,1.2)
ax[1].set_ylabel('Delta (sec)')
ax[1].set_xlabel('Lap Distance (m)')
ax[1].grid(visible=False)

fig.align_labels(axs=ax)

In [ ]:
save_fig(fig=fig, name='delta_analysis', loc='Reel25', trs=True)